# MemGraphRAG ile Bellek Tabanlı GraphRAG: Google Colab Uygulaması

**Ders:** Büyük Dil Modelleri / Bilgi Erişimi Ödevi
**Öğrenci:** _Ad Soyad_ — _Öğrenci No_
**Kaynak kod:** [XMUDeepLIT/MemGraphRAG](https://github.com/XMUDeepLIT/MemGraphRAG) (KDD 2026, [arXiv:2606.00610](https://arxiv.org/abs/2606.00610))

Bu notebook'ta MemGraphRAG reposu Google Colab'a aktarılır, bağımlılıkları kurulur ve seçilen dil modeli sisteme entegre edilir. Ardından repodaki **indeksleme → erişim/soru-cevap** örneği HotpotQA verisinin bir alt kümesi üzerinde uçtan uca çalıştırılır. Sonuçlar klasik (dense) RAG ile karşılaştırılır. Karşılaşılan hatalar, çözümleri ve kısa bir teknik değerlendirme de notebook'ta yer alır.

| Ödev maddesi | Notebook bölümü |
|---|---|
| 1. Repo incelemesi ve temel çalışma mantığı | Bölüm 1 |
| 2–3. Colab'a aktarma, bağımlılık kurulumu | Bölüm 3, 4, 5 |
| 4. Modellerin ve veri setinin hazırlanması | Bölüm 7, 8 |
| 5–6, 9. Dil modeli seçimi, gerekçesi ve entegrasyonu | Bölüm 2, 7 |
| 7–8. Repodaki örneğin çalıştırılması ve sonuçlar | Bölüm 9, 10, 11, 12, 13 |
| 10. Karşılaşılan hatalar ve çözümleri | Bölüm 6 ve 14 |
| 11. Klasik RAG / GraphRAG ile teknik karşılaştırma | Bölüm 15 |

> **Çalıştırma:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin, sonra `Çalışma zamanı → Tümünü çalıştır`. T4 üzerinde toplam süre yaklaşık **30–45 dakikadır**. Sürenin büyük kısmı indeksleme adımında harcanır. İsterseniz Bölüm 2'deki `N_QUESTIONS` değerini düşürerek süreyi kısaltabilirsiniz.

## 1. MemGraphRAG Nedir? Temel Çalışma Mantığı

**Çözülen problem.** Klasik GraphRAG yöntemleri (Microsoft GraphRAG, LightRAG, HippoRAG) bilgi grafını *her metin parçasından (chunk) bağımsız olarak* LLM ile varlık/ilişki çıkararak kurar. Parçalar arasında global bir bakış olmadığı için aynı varlık farklı biçimlerde yazılır, birbiriyle çelişen olgular grafa girer ve gürültülü ilişkiler oluşur. MemGraphRAG bu sorunu, çıkarım sürecine **ortak (global) bir bellek** ve bu bellekle etkileşen **çok ajanlı bir LLM grubu** ekleyerek çözmeye çalışır.

<p align="center"><img src="https://raw.githubusercontent.com/XMUDeepLIT/MemGraphRAG/main/framework.png" width="85%"></p>

### 1.1 Üç katmanlı bellek (`code/src/Memory.py` → `ThreeLayerMemory`)

| Katman | İçerik | Örnek |
|---|---|---|
| **Şema (schema) katmanı** | Soyut ontoloji üçlüleri `(baş tipi, ilişki, kuyruk tipi)` | `(<PERSON>, born in, <GPE>)` |
| **Olgu (fact) katmanı** | Metinden çıkarılmış somut üçlüler `(özne, ilişki, nesne)` | `(chris evans, born in, boston)` |
| **Pasaj (passage) katmanı** | Olguları destekleyen orijinal metin parçaları | `"Christopher Robert Evans (born June 13, 1981)…"` |

Katmanlar çift yönlü indekslerle bağlıdır: her olgu tek bir şemaya ve bir ya da daha fazla pasaja bağlanır.

### 1.2 İndeksleme aşaması (`MemGraphRAG.index_with_memory`)

Makaledeki "ajanlar", kodda farklı görevlere atanmış LLM çağrılarıdır:

1. **Çıkarım ajanı (OpenIE):** Her chunk için önce varlık tanıma (NER), sonra varlıklara koşullu üçlü çıkarımı yapılır (`information_extraction/openie_openai.py`). Üçlüler küçük harfe çevrilip normalize edilir.
2. **Bellek oluşturma:** Üçlüler tekilleştirilerek olgu ve pasaj katmanları kurulur.
3. **Ontoloji/şema ajanı:** Her olgu için, kaynak pasajlar bağlam verilerek bir şema üçlüsü üretilir (`PROMPT['ontology_extraction']`). Böylece somut olgular yeniden kullanılabilir tip-ilişki-tip kalıplarına soyutlanır.
4. **Ontoloji filtreleme:** En düşük frekanslı %20'lik şemalar (yüzdelik mod) ve onlara bağlı olgular elenir. Bu adım seyrek ve gürültülü kalıpları temizler.
5. **Çelişki tespit ajanı:** Aynı (özne, ilişki) çiftine farklı nesne veren olgular aday çelişki olarak seçilir. LLM, pasaj kanıtlarıyla birlikte bu adayların *sert çelişki* olup olmadığına (mutual / temporal / granularity) karar verir (`PROMPT['conflict_detection']`, güven eşiği 0.85).
6. **Çelişki çözüm ajanı:** Çelişen olgular bağlı bileşenlere ayrılır. Her bileşen için LLM kaynak pasajlara bakarak olguyu **korur**, **siler** veya **değiştirir**; örneğin ilişkiye zaman ya da ayrıntı düzeyi bilgisi ekler (`PROMPT['conflict_resolution']`).
7. **Bellekten graf üretimi:** Çelişkileri çözülmüş bellekten üç düğüm tipli bir graf kurulur: **tip**, **varlık** ve **pasaj** düğümleri. Kenarlar varlık–varlık (ilişki), pasaj–varlık, varlık–tip, tip–tip ve gömme benzerliğine dayalı eşanlamlılık kenarlarıdır.

### 1.3 Erişim ve cevap üretimi (`MemGraphRAG.retrieve` + `qa`)

1. Soru BGE ile gömülür ve **tüm olgularla** kosinüs benzerliği hesaplanır.
2. Ham benzerliği eşik değerin (`fact_similarity_threshold`) üzerinde kalan olgular seçilir. İsteğe bağlı olarak LLM tabanlı yeniden sıralama da yapılabilir.
3. Seçilen olguların varlıkları **tohum düğüm** olur ve olgu skoruyla ağırlıklandırılır. Pasaj düğümleri de dense benzerlikle küçük bir ağırlık (0.05) alır.
4. Graf üzerinde **Kişiselleştirilmiş PageRank (PPR)** çalıştırılır ve pasajlar PPR olasılığına göre sıralanır. Bu sayede soruda doğrudan geçmeyen ama graf üzerinden bağlantılı pasajlara da (çok adımlı akıl yürütme) ulaşılır.
5. En iyi `k` pasaj, tek-örnekli (one-shot) bir okuma-anlama istemiyle LLM'e verilir ve `Answer:` satırından cevap alınır.

> Erişim algoritması büyük ölçüde **HippoRAG 2**'den devralınmıştır (README'de de belirtilir). MemGraphRAG'ın asıl katkısı **indeksleme tarafındadır**: global bellek, ontoloji çıkarımı/filtreleme, çelişki tespiti/çözümü ve grafa eklenen tip katmanı.

## 2. Dil Modeli Seçimi, Gerekçesi ve Yapılandırma

MemGraphRAG LLM'e **OpenAI uyumlu bir sohbet API'si** üzerinden erişir (`code/src/llm/openai_gpt.py` → `CacheOpenAI`). Bu nedenle `base_url` ve `model` vermek yeterlidir: OpenAI, Ollama, vLLM veya Groq gibi her uyumlu sunucu kullanılabilir. Gömme (embedding) modeli ise yerel bir Hugging Face modelidir.

### Seçilen model: **Qwen2.5-7B-Instruct** (Ollama üzerinden, 4-bit Q4_K_M)

| Kriter | Değerlendirme |
|---|---|
| **Colab'da çalışabilirlik** | 4-bit ağırlıklar ~4,7 GB; 16K bağlam × 4 paralel istek için KV önbelleği ~3,5 GB ekler (toplam ~9 GB). BGE-large (~1,5 GB) ile birlikte T4'ün 15 GB'ına sığar; Bölüm 7.1'deki kontrol bunu doğrular. |
| **MemGraphRAG ile uyum** | Ollama, OpenAI uyumlu `/v1/chat/completions` uç noktası sunar; repo kodunda değişiklik gerekmez. `localhost` adresinde repo sahte bir API anahtarı atar. |
| **Cevap kalitesi** | Qwen2.5-7B, kendi boyut sınıfında talimat takibi ve **yapılandırılmış JSON üretimi** açısından güçlüdür. Bu kritiktir, çünkü NER, üçlü çıkarımı, ontoloji, çelişki tespiti ve çözümü adımlarının hepsi JSON çıktısı bekler. |
| **Maliyet** | Açık kaynaktır (Apache-2.0) ve ücretsiz T4 GPU'da çalışır. API ücreti ya da kota yoktur; yüzlerce LLM çağrısı maliyetsizdir. |
| **Bağlam uzunluğu** | Model 32K+ bağlamı destekler. Çelişki tespit istemleri pasaj kanıtı içerdiğinden uzundur; Ollama'yı 16K bağlamla başlatıyoruz. |
| **Tekrarlanabilirlik** | `temperature=0`, sabitlenmiş Ollama sürümü (0.40.0) ve yerel model kullanılır. LLM yanıtları SQLite önbelleğinde tutulduğu için yeniden çalıştırmalar aynı sonucu verir. |

**Değerlendirilen alternatifler**

| Model | Neden birincil tercih edilmedi? |
|---|---|
| GPT-4o-mini (OpenAI API) | Makalenin varsayılan modelidir; hızlı ve kalitelidir, ancak ücretli bir API anahtarı gerektirir. Notebook'ta `LLM_BACKEND = "openai"` seçeneğiyle desteklenir. |
| Llama-3.1-8B-Instruct | Benzer boyutta, ancak katı JSON formatına uyumu Qwen2.5'e göre daha zayıftır. Ollama ile `llama3.1:8b` olarak denenebilir. |
| Qwen3-8B (düşünme modu) | Ollama düşünme metnini ayrı bir `reasoning` alanında döndürür, ama düşünme modu varsayılan olarak açıktır ve repo bunu kapatacak bir parametre göndermez. Bu yüzden her çağrıda yüzlerce ek token üretilir; yüzlerce şema/çelişki çağrısında süre katlanır. Uzun düşünme `max_tokens` bütçesini tüketip cevabı boş ya da kesik de bırakabilir. |
| vLLM + tam hassasiyetli model | vLLM, Colab'ın torch 2.11 / Python 3.13 sürümünü değiştirmeye zorlar (bkz. Bölüm 5). 7B fp16 model de T4'e sığmaz. |

**Gömme modeli:** `BAAI/bge-large-en-v1.5`, README'de önerilen ve repoda desteklenen modeldir. Repo modeli adındaki `bge-large-en-v1.5` ifadesinden tanır.

**Chunk tokenizer'ı:** README chunk'lama için gömme modelinin tokenizer'ını önerir. Ancak BGE tokenizer'ı *küçük harfe çeviren (uncased)* bir WordPiece tokenizer'ıdır; metin geri çözüldüğünde (decode) bütün büyük harfler ve bazı karakterler kaybolur. LLM'in varlıkları doğru tanıması için chunk'lamada kayıpsız bir BPE tokenizer olan **Qwen tokenizer'ını** kullanıyoruz (`--tokenizer` parametresi). Chunk boyutu yine 256 token'dır ve BGE'nin 512 token sınırının altında kalır.

> Farklı bir model denemek için aşağıdaki hücrede `LLM_BACKEND` değerini değiştirmeniz yeterlidir. `"openai"` için Colab'ın sol menüsündeki **🔑 Secrets** bölümüne `OPENAI_API_KEY` ekleyin. `"custom"` ile OpenAI uyumlu başka bir servis (ör. Groq, OpenRouter) kullanılabilir. Ancak indeksleme ~600+ çağrı ve ~0,5–1M token gerektirdiği için ücretsiz katmanlar (ör. Groq llama-3.3-70b: 30 istek/dk, 100K token/gün) yetersizdir. Ücretli bir anahtar kullanın ve `MEMORY_MAX_WORKERS` değerini sağlayıcının hız sınırına göre düşürün.

In [ ]:
# ===================== YAPILANDIRMA =====================
import os, sys

LLM_BACKEND = "ollama"                  # "ollama" | "openai" | "custom"

# --- Ollama (varsayılan, ücretsiz, açık kaynak) ---
OLLAMA_MODEL = "qwen2.5:7b-instruct"    # alternatif: "llama3.1:8b", "qwen2.5:3b-instruct" (daha hızlı)
OLLAMA_CONTEXT_LENGTH = 16384           # T4 (<24 GB VRAM) için Ollama varsayılanı yalnızca 4096 token
OLLAMA_NUM_PARALLEL = 4                 # aynı anda işlenecek istek sayısı
OLLAMA_VERSION = "0.40.0"               # tekrarlanabilirlik için sabitlenen Ollama sürümü

# --- OpenAI (opsiyonel, ücretli) ---
OPENAI_MODEL = "gpt-4o-mini"            # anahtar: Colab Secrets -> OPENAI_API_KEY

# --- OpenAI uyumlu başka bir servis (opsiyonel) ---
CUSTOM_BASE_URL = "https://api.groq.com/openai/v1"
CUSTOM_MODEL = "llama-3.3-70b-versatile"     # not: ücretsiz katman hız sınırları bu iş yüküne yetmez
CUSTOM_KEY_SECRET = "GROQ_API_KEY"

# --- Gömme modeli ve chunk'lama ---
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"
CHUNK_TOKENIZER = "Qwen/Qwen2.5-7B-Instruct"   # kayıpsız (büyük/küçük harf korunur)
CHUNK_SIZE, CHUNK_OVERLAP = 256, 32

# --- Veri alt kümesi (HotpotQA) ---
N_QUESTIONS = 20          # soru sayısı (süreyi kısaltmak için 10 yapılabilir)
N_DISTRACTORS = 1         # her soru için eklenecek "çeldirici" (alakasız ama benzer) pasaj sayısı

# --- MemGraphRAG parametreleri ---
MEMORY_MAX_WORKERS = 8           # bellek aşamalarında paralel LLM isteği (repo varsayılanı 20)
CONFLICT_MAX_RELATED = 10        # her olgu için karşılaştırılacak en fazla aday (repo varsayılanı 20)
FACT_THRESHOLD = 0.4             # README'deki değer
TOP_K = 5                        # erişilen / LLM'e verilen pasaj sayısı

# --- Yollar ---
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content" if IN_COLAB else os.getcwd()
REPO_URL = "https://github.com/XMUDeepLIT/MemGraphRAG.git"
REPO_COMMIT = "cd6fabd"          # tekrarlanabilirlik için sabitlenen commit
REPO_DIR = f"{BASE_DIR}/MemGraphRAG"
CORPUS_DIR = f"{REPO_DIR}/datasets/corpus"          # README'deki yol
LOG_DIR = f"{BASE_DIR}/logs"
os.makedirs(LOG_DIR, exist_ok=True)

if LLM_BACKEND == "ollama":
    LLM_NAME, LLM_BASE_URL = OLLAMA_MODEL, "http://localhost:11434/v1"
elif LLM_BACKEND == "openai":
    LLM_NAME, LLM_BASE_URL = OPENAI_MODEL, "https://api.openai.com/v1"
elif LLM_BACKEND == "custom":
    LLM_NAME, LLM_BASE_URL = CUSTOM_MODEL, CUSTOM_BASE_URL
else:
    raise ValueError(LLM_BACKEND)
# Her LLM için ayrı çıktı klasörü (memory.json, graf ve sonuçlar modeller arasında karışmasın)
RUN_TAG = LLM_NAME.replace("/", "_").replace(":", "_")
SAVE_DIR = f"{REPO_DIR}/outputs/corpus_{RUN_TAG}"
RESULTS_PATH = f"{REPO_DIR}/results/corpus_{RUN_TAG}/qa_results.json"
print(f"LLM: {LLM_NAME} @ {LLM_BASE_URL}\nEmbedding: {EMBEDDING_MODEL}\nÇıktı klasörü: {SAVE_DIR}\nColab: {IN_COLAB}")

## 3. Ortam Kontrolü

Kontrol edilenler: GPU, Python sürümü ve Colab'da hazır kurulu gelen `torch` / `transformers` sürümleri. Colab Ağustos 2026 sonundan beri **Python 3.13** kullanıyor (sürümler aşağıda yazdırılır); Bölüm 5'teki ilk hatanın kaynağı da bu.

In [ ]:
import platform, shutil, subprocess
import torch, transformers, numpy

print("Python      :", platform.python_version())
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("numpy       :", numpy.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name} ({p.total_memory / 1024**3:.1f} GB)")
else:
    print("UYARI: GPU bulunamadı. Çalışma zamanı türünü T4 GPU olarak değiştirin.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True).stdout)

## 4. Repository'nin Colab'a Aktarılması

Repo klonlanır ve tekrarlanabilirlik için bu notebook'un test edildiği commit'e (`cd6fabd`) sabitlenir.

In [ ]:
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout -q {REPO_COMMIT} && git -C {REPO_DIR} log --oneline -1
os.chdir(REPO_DIR)
!find . -path ./.git -prune -o -name "__pycache__" -prune -o -type f -name "*.py" -print | sort | head -60
!ls -la dataset/*

## 5. Bağımlılıkların Kurulumu

### 5.1 `requirements.txt` neden doğrudan kurulamıyor? (Karşılaşılan Hata #1)

Repodaki `requirements.txt` eski ve katı sürüm sabitlemeleri içerir: `vllm==0.6.6.post1`, `torch==2.5.1`, `transformers==4.45.2`. `vllm 0.6.6.post1`, Python 3.13 için wheel'i olmayan `torchvision==0.20.1`'i şart koşar. Bu yüzden `pip install -r requirements.txt` Colab'da şu hatayla durur:

```
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1 (from vllm) (from versions: 0.21.0, ... 0.29.1)
ERROR: No matching distribution found for torchvision==0.20.1
```

Kurulum başarılı olsaydı bile Colab'ın torch/transformers sürümlerini düşürüp diğer paketleri bozardı.

**Çözüm:** Kodu inceleyerek *gerçekten import edilen* paketleri belirledik. `vllm` yalnızca kullanılmayan "offline OpenIE" modunda gerekiyor; `litellm`, `boto3` ve `gritlm` online akışta kullanılmıyor. Colab'da hazır gelen `torch`, `transformers`, `numpy`, `pandas`, `openai`, `pyarrow` ve `accelerate` paketleri olduğu gibi kullanılır. Yalnızca eksik olan hafif paketler kurulur.

Aşağıdaki hücre hatayı notebook içinde **göstermek** için `pip`'in `--dry-run` modunu çalıştırır. Bu mod hiçbir şeyi kurmaz ve 1–2 dakika sürebilir; atlamak için `SHOW_REQUIREMENTS_ERROR = False` yapın.

In [ ]:
SHOW_REQUIREMENTS_ERROR = True
if SHOW_REQUIREMENTS_ERROR:
    !timeout 300 {sys.executable} -m pip install --dry-run -q -r {REPO_DIR}/requirements.txt 2>&1 | grep -E "Could not find a version|No matching distribution" | cut -c1-250

In [ ]:
# Online MemGraphRAG akışının gerçekten ihtiyaç duyduğu paketler (çoğu Colab'da zaten kurulu)
!{sys.executable} -m pip install -q "python-igraph>=0.11" tenacity filelock pyarrow accelerate openai
import importlib.metadata as md_
for pkg in ["python-igraph", "openai", "tenacity", "filelock", "pyarrow", "accelerate", "transformers", "torch"]:
    print(f"{pkg:15s} {md_.version(pkg)}")

## 6. Kaynak Kod Düzeltmeleri (Yamalar)

Notebook hazırlanırken repo **Colab ile aynı sürümlerde** (Python 3.13, torch 2.11, transformers 5.x) çalıştırıldı. Bu sırada aşağıdaki hatalar alındı ya da kod incelemesiyle tespit edildi. Düzeltmeler küçük ve hedefli metin değişiklikleri olarak uygulanır. Hücre her yamanın repoda bulunduğunu doğrular, iki kez çalıştırılsa da sorun çıkarmaz ve sonunda `git diff` ile değişiklikleri gösterir.

| Yama | Hata / belirti | Neden | Düzeltme |
|---|---|---|---|
| **P1** | `ModuleNotFoundError: No module named 'gritlm'` (ilk `import` anında) | `embedding_model/__init__.py` GritLM modülünü koşulsuz içe aktarıyor. `gritlm` kurulumu ise `mteb`, `wandb`, `datasets` gibi ağır paketleri çekiyor. | İçe aktarmayı `try/except ImportError` içine al (BGE kullanıyoruz) |
| **P2** | `RuntimeError: index_with_memory failed during schema extraction: Expecting value: line 1 column 1 (char 0)` | Ontoloji/çelişki adımlarında tek bir JSON olmayan LLM yanıtı (açıklama metni, kesik çıktı, liste, `"confidence": "high"`) tüm indekslemeyi durduruyor. Yanıt SQLite önbelleğine yazıldığı için yeniden çalıştırınca **aynı hata tekrar ediyor**. | `_memory_json` ayrıştırılamayan çıktıda boş sözlük döndürür, uyarı loglar ve alan tiplerini düzeltir |
| **P3** | Çökme yok, mantık hatası | `conflicting_triple_ids` listesindeki kimlikler `conflict_min_confidence=0.85` eşiği kontrol edilmeden çelişki sayılıyor (varsayılan güven 0.6 atanıyor). | Yalnızca eşiği geçen "sert çelişki" kayıtları çözüme gönderilir; P3b, LLM `triple1`/`triple2` sırasını ters yazsa bile eşleşmeyi sağlar |
| **P4** | Çökme yok, mantık hatası | Çözüm adımında LLM'in döndürdüğü herhangi bir olgu kimliği kabul ediliyor; uydurulmuş bir kimlik ilgisiz bir olguyu silebiliyor. | Yalnızca ilgili çelişki bileşenindeki olgulara işlem uygulanır |
| **P5** | Erişimde `AssertionError` riski | Çözümde *değiştirilen* (modified) olgular normalize edilmiyor (ör. `"Alice"`). Erişim ise varlıkları küçük harfle arıyor; düğüm bulunamıyor. | Değiştirilen olgulara OpenIE ile aynı `text_processing` normalizasyonu uygulanır |
| **P6** | `AssertionError` (`get_top_k_weights`, `assert np.count_nonzero(...) == len(...)`) | Tüm olgular eşiği geçtiğinde min-max normalizasyonu en düşük olguya 0 ağırlık veriyor ve sayılar eşleşmiyor (HippoRAG'den devralınmış). | `assert` yerine debug logu |
| **P7** | Yüzlerce satırlık `HTTP Request: POST ...` logu | `httpx` logları yalnızca OpenIE sırasında susturuluyor. | `httpx` log seviyesi WARNING yapılır |
| **P8** | `TypeError: object of type 'int' has no len()` (`save_openie_results`) | NER yanıtı `["Apple", 2001]` gibi tırnaksız bir sayı içerirse varlık listesi metin olmayan öğe taşıyor ve OpenIE aşaması çöküyor. | Varlıklar metne çevrilir, liste/sözlük gibi öğeler atlanır |

P1, P2 ve P6 yerel testlerde **gerçekten karşılaşılan** çökmelerdir. P3, P4, P5 ve P8 kod incelemesinde bulunan ve sahte (mock) LLM yanıtlarıyla doğrulanan hatalardır.

In [ ]:
# ---- Kaynak koda uygulanan düzeltmeler (idempotent: iki kez çalıştırmak sorun çıkarmaz) ----
import pathlib

PATCHES = [
    # (dosya, eski metin, yeni metin, açıklama)
    ("code/src/embedding_model/__init__.py",
     "from .GritLM import GritLMEmbeddingModel\n",
     "try:\n"
     "    from .GritLM import GritLMEmbeddingModel\n"
     "except ImportError:  # [Colab yaması P1] gritlm yalnızca GritLM gömme modeli için gerekli\n"
     "    GritLMEmbeddingModel = None\n",
     "P1 – gritlm opsiyonel içe aktarma (ModuleNotFoundError: No module named 'gritlm')"),

    ("code/src/MemGraphRAG.py",
     "        try:\n"
     "            return json.loads(cleaned)\n"
     "        except json.JSONDecodeError:\n"
     "            match = re.search(r\"\\{.*\\}\", cleaned, re.DOTALL)\n"
     "            candidate = match.group(0) if match else cleaned\n"
     "            return json.loads(fix_broken_generated_json(candidate))\n",
     "        try:\n"
     "            try:\n"
     "                out = json.loads(cleaned)\n"
     "            except json.JSONDecodeError:\n"
     "                match = re.search(r\"\\{.*\\}\", cleaned, re.DOTALL)\n"
     "                candidate = match.group(0) if match else cleaned\n"
     "                out = json.loads(fix_broken_generated_json(candidate))\n"
     "        except Exception:  # [Colab yaması P2] tek bir bozuk LLM yanıtı tüm indekslemeyi durdurmasın\n"
     "            logger.warning(\"Unparseable memory-stage LLM output skipped: %r\", cleaned[:200])\n"
     "            out = {}\n"
     "        if not isinstance(out, dict):\n"
     "            out = {}\n"
     "        for key in (\"ontology_triples\", \"conflicts\", \"conflicting_triple_ids\", \"resolved_triples\"):\n"
     "            if key in out and not isinstance(out[key], list):\n"
     "                out[key] = []\n"
     "            if key in out and key != \"conflicting_triple_ids\":\n"
     "                out[key] = [row for row in out[key] if isinstance(row, dict)]\n"
     "        for entry in out.get(\"conflicts\", []):\n"
     "            try:\n"
     "                entry[\"confidence\"] = float(entry.get(\"confidence\", 0.0) or 0.0)\n"
     "            except (TypeError, ValueError):\n"
     "                entry[\"confidence\"] = 0.0\n"
     "            for flag in (\"is_hard_conflict\", \"needs_resolution\"):\n"
     "                if isinstance(entry.get(flag), str):\n"
     "                    entry[flag] = entry[flag].strip().lower() == \"true\"\n"
     "        return out\n",
     "P2 – bellek aşamalarında bozuk/JSON olmayan LLM çıktısına dayanıklı ayrıştırma"),

    ("code/src/MemGraphRAG.py",
     "            for other_id in result[\"conflicting_related_ids\"]:\n"
     "                pair = tuple(sorted((target_id, int(other_id))))\n",
     "            for other_id in result[\"conflicting_related_ids\"]:\n"
     "                if int(other_id) not in detailed:  # [Colab yaması P3] conflict_min_confidence eşiğini uygula\n"
     "                    continue\n"
     "                pair = tuple(sorted((target_id, int(other_id))))\n",
     "P3 – çelişki güven eşiğinin (0.85) atlanmasını engelle"),

    ("code/src/MemGraphRAG.py",
     "                for field in (\"triple2\", \"other_triple\"):\n",
     "                for field in (\"triple2\", \"other_triple\", \"triple1\"):  # [Colab yaması P3b] sıra ters olsa da eşle\n",
     "P3b – çelişki kaydında triple1/triple2 sırası ters olsa da eşleştir"),

    ("code/src/MemGraphRAG.py",
     "        for result in component_results:\n"
     "            for row in result.get(\"resolved_triples\", []):\n",
     "        for result in component_results:\n"
     "            component_ids = {int(x) for x in result.get(\"component_fact_ids\", [])}  # [Colab yaması P4]\n"
     "            for row in result.get(\"resolved_triples\", []):\n",
     "P4a – çözüm adımında yalnızca ilgili bileşendeki olgular değiştirilebilsin"),

    ("code/src/MemGraphRAG.py",
     "                    fact_idx = candidate if candidate in facts else None\n",
     "                    fact_idx = candidate if candidate in facts and candidate in component_ids else None\n",
     "P4b – LLM'in uydurduğu/bileşen dışı olgu kimliklerini yok say"),

    ("code/src/MemGraphRAG.py",
     "                    fact_idx = matches[0] if len(matches) == 1 else None\n",
     "                    fact_idx = matches[0] if len(matches) == 1 and matches[0] in component_ids else None\n",
     "P4c – metin eşleşmesiyle bulunan olgu da bileşen içinde olmalı"),

    ("code/src/MemGraphRAG.py",
     "                new_fact[\"content\"] = list(map(str, resolved))\n",
     "                new_fact[\"content\"] = [text_processing(resolved[0]),  # [Colab yaması P5] OpenIE ile aynı normalizasyon\n"
     "                                        \" \".join(str(resolved[1]).lower().split()),\n"
     "                                        text_processing(resolved[2])]\n",
     "P5 – değiştirilen (modified) olguları küçük harfe normalize et"),

    ("code/src/MemGraphRAG.py",
     "        assert np.count_nonzero(all_phrase_weights) == len(linking_score_map.keys())\n",
     "        if np.count_nonzero(all_phrase_weights) != len(linking_score_map.keys()):  # [Colab yaması P6]\n"
     "            logger.debug(\"Some linked phrases have zero weight or are not graph nodes; continuing.\")\n",
     "P6 – erişimdeki AssertionError (get_top_k_weights) yerine uyarı"),

    ("code/index.py",
     "    logging.basicConfig(level=logging.INFO)\n",
     "    logging.basicConfig(level=logging.INFO)\n"
     "    logging.getLogger(\"httpx\").setLevel(logging.WARNING)  # [Colab yaması P7] her LLM isteğini loglama\n",
     "P7 – httpx istek loglarını sustur (yüzlerce satırlık çıktı)"),
    ("code/src/information_extraction/openie_openai.py",
     "            unique_entities = list(dict.fromkeys(extracted_entities))\n",
     "            unique_entities = list(dict.fromkeys(  # [Colab yaması P8] sayı/dizi gibi metin olmayan varlıklar\n"
     "                str(e) for e in extracted_entities if isinstance(e, (str, int, float))))\n",
     "P8 – NER çıktısındaki metin olmayan varlıkları (ör. 2001) metne çevir"),
]

def apply_patches(repo_dir):
    rows = []
    for rel, old, new, desc in PATCHES:
        path = pathlib.Path(repo_dir) / rel
        text = path.read_text(encoding="utf-8")
        if new in text:
            status = "zaten uygulanmış"
        elif text.count(old) == 1:
            path.write_text(text.replace(old, new), encoding="utf-8")
            status = "uygulandı"
        else:
            raise RuntimeError(f"Yama bulunamadı ({desc}) -> {rel}. Repo farklı bir commit'te olabilir.")
        rows.append((desc, rel, status))
    return rows


import pandas as pd
patch_report = pd.DataFrame(apply_patches(REPO_DIR), columns=["Yama", "Dosya", "Durum"])
display(patch_report)
!git -C {REPO_DIR} diff --stat

## 7. Dil Modelinin Hazırlanması ve Entegrasyonu

`LLM_BACKEND == "ollama"` iken bu hücre şunları yapar:

1. Ollama'yı kurar. Güncel Linux sürümleri `.tar.zst` arşivi olarak dağıtıldığı için önce `zstd` kurulmalıdır; aksi hâlde kurulum betiği `requires zstd` hatası verir. Tekrarlanabilirlik için Ollama sürümü 0.40.0'a sabitlenir.
2. Sunucuyu 16K bağlam ve 4 paralel istekle arka planda başlatır. T4 için Ollama'nın varsayılan bağlamı **4096 token**'dır. Pasaj kanıtı içeren çelişki tespit istemleri bunu aşınca Ollama istemin başını *sessizce keser*; sistem talimatı kaybolur ve bozuk JSON üretilir.
3. `qwen2.5:7b-instruct` modelini indirir (~4.7 GB).

`openai` / `custom` seçildiyse API anahtarı Colab Secrets'tan okunur.

In [ ]:
import time, json, subprocess, urllib.request

def wait_for(url, timeout=120):
    for _ in range(timeout):
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                return r.read().decode()
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"Sunucu yanıt vermedi: {url}")

if LLM_BACKEND == "ollama":
    if shutil.which("ollama") is None:
        !apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1 || echo "UYARI: zstd kurulamadı"
        !curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh
        assert shutil.which("ollama"), "Ollama kurulamadı; yukarıdaki kurulum çıktısını kontrol edin (zstd?)"
    try:
        wait_for("http://localhost:11434/api/version", timeout=2)
        print("Ollama zaten çalışıyor.")
    except RuntimeError:
        env = dict(os.environ,
                   OLLAMA_HOST="127.0.0.1:11434",
                   OLLAMA_CONTEXT_LENGTH=str(OLLAMA_CONTEXT_LENGTH),
                   OLLAMA_NUM_PARALLEL=str(OLLAMA_NUM_PARALLEL),
                   OLLAMA_KEEP_ALIVE="-1")
        ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                                       stdout=open(f"{LOG_DIR}/ollama.log", "w"), stderr=subprocess.STDOUT)
    print("Ollama sürümü:", wait_for("http://localhost:11434/api/version"))
    !ollama pull {OLLAMA_MODEL} 2>&1 | tail -n 2
    assert OLLAMA_MODEL in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout, "Model indirilemedi"
    os.environ["OPENAI_API_KEY"] = "ollama"      # yerel sunucu anahtar istemez, istemci boş olmamasını ister
else:
    from google.colab import userdata
    secret = "OPENAI_API_KEY" if LLM_BACKEND == "openai" else CUSTOM_KEY_SECRET
    os.environ["OPENAI_API_KEY"] = userdata.get(secret)
    print(f"{secret} Colab Secrets'tan okundu.")

### 7.1 Entegrasyon testi

MemGraphRAG'ın kullandığı istemciyle aynı şekilde (OpenAI SDK, `temperature=0`) kısa bir NER isteği gönderilir. Model geçerli JSON döndürüyorsa entegrasyon tamamdır.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=LLM_BASE_URL, api_key=os.environ["OPENAI_API_KEY"])
t0 = time.time()
resp = client.chat.completions.create(
    model=LLM_NAME, temperature=0, max_tokens=200,
    messages=[{"role": "system", "content": "Your task is to extract named entities from the given paragraph. Respond with a JSON list of entities."},
              {"role": "user", "content": "Radio City is India's first private FM radio station and was started on 3 July 2001."}])
print(resp.choices[0].message.content)
print(f"Süre: {time.time() - t0:.1f} sn | token kullanımı: {resp.usage.prompt_tokens} girdi / {resp.usage.completion_tokens} çıktı")
if LLM_BACKEND == "ollama":
    !ollama ps
    # Ayarların gerçekten uygulandığını doğrula: 16K bağlam ve modelin tamamen GPU'da olması
    m = json.loads(urllib.request.urlopen("http://localhost:11434/api/ps").read())["models"][0]
    ctx, vram, size = m.get("context_length"), m.get("size_vram", 0), m.get("size", 1)
    print(f"Bağlam uzunluğu: {ctx} | GPU'daki boyut: {vram / 1e9:.1f} / {size / 1e9:.1f} GB")
    if ctx is not None:
        assert ctx >= OLLAMA_CONTEXT_LENGTH, "Ollama eski ayarlarla çalışıyor: !pkill -f 'ollama serve' çalıştırıp Bölüm 7 hücresini tekrarlayın"
    assert vram >= 0.95 * size, "Model tamamen GPU'da değil (CPU'ya taşmış); çalışma zamanı türünün T4 GPU olduğunu kontrol edin"

## 8. Veri Setinin Hazırlanması

README'deki örnek komutlar `datasets/corpus/corpus.txt` ve `datasets/corpus/small-questions.json` dosyalarını kullanır. **Bu dosyalar repoda yoktur** (Karşılaşılan Hata #2). Repoda yalnızca `dataset/` (tekil) klasöründe `hotpotqa`, `musique`, `2wikimutlhopqa` (yazım hatalı klasör adıyla) ve `medical` verileri bulunur. README yolu kullanılınca `FileNotFoundError` alınır. Ayrıca `code/run_index.sh` ve `code/run_retrieval_test.sh` betikleri yazarın kendi bilgisayarındaki `/home/wcj/...` yollarını içerir ve tanımsız `$PYTHON` değişkeni nedeniyle `PYTHON: unbound variable` hatasıyla durur.

**Çözüm:** README'nin beklediği iki dosyayı repodaki `dataset/hotpotqa/hotpotqa.json` verisinden üretiyoruz:

- **Sorular:** HotpotQA'nın ilk `N_QUESTIONS` sorusu. Hepsi çok adımlı (multi-hop) ve "hard" zorluktadır.
- **Korpus:** Her sorunun iki *altın (gold)* destekleyici pasajı ve aynı sorunun bağlamından `N_DISTRACTORS` çeldirici pasaj. Çeldiriciler soruyla konu olarak benzer ama cevabı içermez; bu yüzden erişimi zorlaştırır.
- **Soru dosyası:** README'nin "tipe göre gruplanmış" formatında yazılır (`bridge` / `comparison`).
- **Karıştırma:** Pasajlar sabit tohumla (seed=0) karıştırılır. Karıştırılmazsa aynı sorunun iki altın pasajı yan yana yazılır. 256 token'lık bir chunk 2–3 HotpotQA pasajı aldığından, ikisi çoğunlukla aynı chunk'a düşer ve çok adımlı erişim farkı ölçülemez hâle gelir.

Tüm `hotpotqa.txt` (~5.6 MB, ~5.800 chunk) Colab'da 7B bir modelle saatler sürer. Alt küme seçmemizin nedeni budur.

In [ ]:
import json, ast, collections, random

def build_hotpotqa_subset(src_json, n_questions, n_distractors):
    data = json.load(open(src_json, encoding="utf-8"))
    as_list = lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    passages, groups = {}, collections.defaultdict(list)
    for i, q in enumerate(data[:n_questions]):
        gold = {title for title, _ in as_list(q["supporting_facts"])}
        n_added = 0
        for title, sentences in as_list(q["context"]):
            if title in gold:
                passages[title] = "".join(sentences).strip()
            elif n_added < n_distractors:
                passages.setdefault(title, "".join(sentences).strip())
                n_added += 1
        groups[q.get("type", "type1")].append({
            "id": q.get("_id", f"q{i}"), "question": q["question"], "answer": q["answer"],
            "evidence": sorted(gold), "supporting_facts": as_list(q["supporting_facts"])})
    items = list(passages.items())
    random.Random(0).shuffle(items)   # aynı sorunun pasajları yan yana (aynı chunk'ta) olmasın
    corpus = "\n\n".join(f"{title}\n{text}" for title, text in items)
    questions = [{"source": "hotpotqa", "questions": dict(groups)}]
    return corpus, questions, passages

corpus_text, questions_json, passages = build_hotpotqa_subset(
    f"{REPO_DIR}/dataset/hotpotqa/hotpotqa.json", N_QUESTIONS, N_DISTRACTORS)
os.makedirs(CORPUS_DIR, exist_ok=True)
open(f"{CORPUS_DIR}/corpus.txt", "w", encoding="utf-8").write(corpus_text)
json.dump(questions_json, open(f"{CORPUS_DIR}/small-questions.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

QUESTIONS = [dict(q, question_type=t) for t, qs in questions_json[0]["questions"].items() for q in qs]
print(f"Pasaj sayısı : {len(passages)}  |  korpus: {len(corpus_text):,} karakter")
print(f"Soru sayısı  : {len(QUESTIONS)}  ->  " + ", ".join(f"{t}: {len(v)}" for t, v in questions_json[0]['questions'].items()))
display(pd.DataFrame([{k: q[k] for k in ("question_type", "question", "answer", "evidence")} for q in QUESTIONS]).head(8))
print(corpus_text[:600], "...")

## 9. İndeksleme: Bellek Tabanlı Graf Oluşturma (Repo Örneği — Adım 1)

README'deki `code/index.py` komutu, seçilen modellerle çalıştırılır. README'den farklı olan parametreler:

- `--tokenizer`: Qwen tokenizer'ı (büyük harfler korunur, bkz. Bölüm 2).
- `--artifact-mode detailed`: Ara aşamaların (ontoloji filtreleme, çelişki tespiti, çelişki çözümü) çıktılarını da kaydeder; Bölüm 10'da incelenir.
- `--memory-max-workers 8` ve `--conflict-max-related 10`: Tek GPU'lu yerel sunucuda kuyruk ve bağlam uzunluğunu makul tutar.

- `--force-openie-from-scratch`: OpenIE sonuçlarını yeniden kurar. Repo, hata veren chunk'ları boş sonuçla kaydedip bir daha denemediği için bu bayrak gereklidir.

LLM yanıtları `outputs/corpus_<model>/llm_cache/*.sqlite` dosyasında önbelleğe alınır. Hücre yarıda kesilirse, yeniden çalıştırıldığında tamamlanmış çağrılar tekrarlanmaz. Repo yeni chunk'ları eski indeksin üzerine eklediği için, `N_QUESTIONS` veya chunk ayarları değişince hücre eski indeksi kendisi siler (LLM önbelleği korunur).

In [ ]:
os.chdir(REPO_DIR)
import hashlib
# Chunk deposu yalnızca ekleme yapar: korpus veya chunk ayarları değiştiyse eski indeksi sil (LLM önbelleği korunur)
fp = hashlib.md5(f"{corpus_text}|{CHUNK_TOKENIZER}|{CHUNK_SIZE}|{CHUNK_OVERLAP}".encode()).hexdigest()
fp_path = f"{SAVE_DIR}/.corpus_fingerprint"
if os.path.isdir(SAVE_DIR) and (not os.path.isfile(fp_path) or open(fp_path).read() != fp):
    for name in os.listdir(SAVE_DIR):
        if name != "llm_cache":
            path = os.path.join(SAVE_DIR, name)
            shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
    print("Korpus/chunk ayarları değişti: eski indeks silindi (LLM önbelleği korundu).")
os.makedirs(SAVE_DIR, exist_ok=True)
open(fp_path, "w").write(fp)

index_cmd = (f'PYTHONUNBUFFERED=1 {sys.executable} code/index.py '
             f'--corpus {CORPUS_DIR}/corpus.txt --save-dir {SAVE_DIR} '
             f'--llm-name "{LLM_NAME}" --llm-base-url "{LLM_BASE_URL}" '
             f'--embedding-model "{EMBEDDING_MODEL}" --tokenizer "{CHUNK_TOKENIZER}" '
             f'--chunk-size {CHUNK_SIZE} --chunk-overlap {CHUNK_OVERLAP} --artifact-mode detailed '
             f'--memory-max-workers {MEMORY_MAX_WORKERS} --conflict-max-related {CONFLICT_MAX_RELATED} '
             f'--force-openie-from-scratch')   # hatalı chunk'lar yeniden denenir; başarılı çağrılar önbellekten gelir
print(index_cmd, "\n")
t0 = time.time()
!{index_cmd} 2>&1 | tee {LOG_DIR}/index.log
INDEX_SECONDS = time.time() - t0
assert "Index completed in" in open(f"{LOG_DIR}/index.log", encoding="utf-8", errors="ignore").read(), \
    "İndeksleme başarısız: logs/index.log dosyasını kontrol edin"
print(f"\nİndeksleme süresi: {INDEX_SECONDS / 60:.1f} dakika")

## 10. İndeksleme Çıktılarının İncelenmesi

### 10.1 Üretilen dosyalar ve OpenIE örneği

In [ ]:
!cd {SAVE_DIR} && find . -maxdepth 3 -not -path "*/llm_cache*" | sort

LLM_LABEL, EMB_LABEL = LLM_NAME.replace("/", "_"), EMBEDDING_MODEL.replace("/", "_")
openie = json.load(open(f"{SAVE_DIR}/openie_results_ner_{LLM_LABEL}.json", encoding="utf-8"))
docs = openie["docs"]
empty = sum(1 for d in docs if not d["extracted_triples"])
print(f"\nChunk sayısı: {len(docs)} | üçlü çıkarılamayan chunk: {empty} | "
      f"toplam üçlü: {sum(len(d['extracted_triples']) for d in docs)}")
d0 = max(docs, key=lambda d: len(d["extracted_triples"]))
print("\nÖrnek chunk:\n", d0["passage"][:400], "...")
print("\nVarlıklar:", d0["extracted_entities"][:15])
print("Üçlüler (normalize edilmiş):")
for t in d0["extracted_triples"][:10]:
    print("  ", t["processed_triple"])

### 10.2 Üç katmanlı belleğin aşama aşama değişimi

OpenIE'den gelen olguların her aşamada nasıl elendiği/değiştiği aşağıda görülür. MemGraphRAG'ı klasik GraphRAG'tan ayıran "bellek küratörlüğü" bu tabloda somutlaşır.

In [ ]:
import matplotlib.pyplot as plt

def mem_stats(name):
    path = f"{SAVE_DIR}/{name}"
    if not os.path.isfile(path):
        return None
    m = json.load(open(path, encoding="utf-8"))
    return {"şema": len(m["schema_layer"]), "olgu": len(m["fact_layer"]), "pasaj": len(m["passage_layer"]),
            "şemasız olgu": sum(1 for f in m["fact_layer"] if int(f.get("schema_idx", -1)) < 0)}

stages = [("1) OpenIE sonrası ilk bellek", "initial_memory.json"),
          ("2) Şema çıkarımı sonrası", "initial_memory_with_schema.json"),
          ("3) Ontoloji filtreleme sonrası", "memory_after_ontology_filtering.json"),
          ("4) Çelişki çözümü sonrası (nihai)", "memory.json")]
stage_df = pd.DataFrame([dict(aşama=s, **(mem_stats(f) or {})) for s, f in stages]).set_index("aşama")
display(stage_df)

ax = stage_df["olgu"].plot.barh(figsize=(8, 2.8), color=["#9db4c0", "#5c6b73", "#c2dfe3", "#253237"])
ax.invert_yaxis(); ax.set_xlabel("olgu (fact) sayısı"); ax.set_title("Bellek küratörlüğü: aşamalara göre olgu sayısı")
for i, v in enumerate(stage_df["olgu"]):
    ax.text(v, i, f" {v}", va="center")
plt.tight_layout(); plt.show()

final_mem = json.load(open(f"{SAVE_DIR}/memory.json", encoding="utf-8"))
top_schemas = sorted(final_mem["schema_layer"], key=lambda s: -s["frequency"])[:12]
print("En sık şemalar (tip, ilişki, tip) -> olgu sayısı")
for s in top_schemas:
    print(f"  {tuple(s['content'])}  ->  {s['frequency']}")

### 10.3 Çelişki tespiti ve çözümü

Çelişki tespit ajanının bulduğu sert çelişkiler ve çözüm ajanının kararları (koru / sil / değiştir):

In [ ]:
det = json.load(open(f"{SAVE_DIR}/conflict_detection_results.json", encoding="utf-8"))
res = json.load(open(f"{SAVE_DIR}/conflict_resolution_results.json", encoding="utf-8"))
print("Tespit özeti :", det["summary"])
print("Çözüm özeti  :", res["summary"])
rows = [{"hedef olgu": c["target_triple"], "diğer olgu": c["other_triple"], "tip": c["conflict_type"],
         "güven": c["confidence"], "gerekçe": str(c["reason"])[:120]} for c in det["conflict_items"][:10]]
if rows:
    display(pd.DataFrame(rows))
acts = [{"olgu id": a["fact_id"], "karar": a["action"], "yeni üçlü": a["resolved_triple"], "gerekçe": a["reason"][:120]}
        for a in res["applied_actions"][:15]]
if acts:
    display(pd.DataFrame(acts))
elif rows:
    print("Çelişkiler tespit edildi, ancak çözüm ajanı geçerli bir karar döndürmedi (bkz. P2/P4).")
else:
    print("Bu korpusta çözüm gerektiren sert çelişki bulunmadı.")

### 10.4 Bellekten üretilen graf

Grafın düğüm/kenar istatistikleri ve bir sorunun varlıkları etrafındaki alt graf. Renkler: **tip** (turuncu), **varlık** (mavi), **pasaj** (gri).

In [ ]:
import networkx as nx

mg = json.load(open(f"{SAVE_DIR}/graph_from_memory/memory_graph.json", encoding="utf-8"))
display(pd.Series(mg["stats"], name="değer").to_frame())

G = nx.Graph()
for n in mg["nodes"]:
    G.add_node(n["id"], layer=n["layer"], label=str(n.get("label", n["content"]))[:28])
for e in mg["edges"]:
    if e["type"] != "entity_similarity":          # görselleştirmede eşanlamlılık kenarlarını gizle
        G.add_edge(e["source"], e["target"], type=e["type"])

def draw_neighbourhood(keywords, max_nodes=45):
    entities = [n for n, a in G.nodes(data=True) if a["layer"] == "entity"]
    seeds = [n for n in entities if any(k.lower() in G.nodes[n]["label"].lower() for k in keywords)]
    seeds = sorted(seeds, key=G.degree, reverse=True)[:3] or sorted(entities, key=G.degree, reverse=True)[:1]
    if not seeds:
        print("Grafta varlık düğümü yok."); return
    ordered = list(dict.fromkeys(seeds + [nb for s in seeds for nb in list(G.neighbors(s))[:15]]))
    sub = G.subgraph(ordered[:max_nodes])
    colors = {"type": "#f28e2b", "entity": "#4e79a7", "passage": "#bab0ac"}
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(sub, seed=7, k=0.6)
    nx.draw_networkx(sub, pos, node_color=[colors[sub.nodes[n]["layer"]] for n in sub],
                     labels={n: sub.nodes[n]["label"] if sub.nodes[n]["layer"] != "passage" else "pasaj" for n in sub},
                     font_size=8, node_size=420, edge_color="#cccccc")
    plt.title("Alt graf: " + ", ".join(G.nodes[s]["label"] for s in seeds)); plt.axis("off"); plt.show()

# İlk sorunun altın pasaj başlıklarındaki kelimelerle eşleşen varlıkları (yoksa en yüksek dereceli varlığı) çiz
keywords = [w for title in QUESTIONS[0]["evidence"] for w in title.replace("(", " ").replace(")", " ").split() if len(w) > 3]
draw_neighbourhood(keywords)

## 11. Erişim ve Soru-Cevap (Repo Örneği — Adım 2)

README'deki `code/retrieval_dataset_test.py` komutu çalıştırılır. Erişim parametreleri README ile aynıdır: `--skip-fact-rerank true`, `--fact-similarity-threshold 0.4`, `--use-raw-threshold-filter true`. Ek olarak, karşılaştırmanın anlamlı olması için hem erişilen hem LLM'e verilen pasaj sayısı `TOP_K=5` alınır. Varsayılan 10 pasaj, küçük korpusun önemli bir kısmına denk gelir.

In [ ]:
qa_cmd = (f'PYTHONUNBUFFERED=1 {sys.executable} code/retrieval_dataset_test.py '
          f'--questions {CORPUS_DIR}/small-questions.json --save-dir {SAVE_DIR} --output {RESULTS_PATH} '
          f'--llm-name "{LLM_NAME}" --llm-base-url "{LLM_BASE_URL}" --embedding-model "{EMBEDDING_MODEL}" '
          f'--question-type all --sample-num 0 --skip-fact-rerank true '
          f'--fact-similarity-threshold {FACT_THRESHOLD} --use-raw-threshold-filter true '
          f'--retrieval-top-k {TOP_K} --qa-top-k {TOP_K}')
print(qa_cmd, "\n")
if os.path.exists(RESULTS_PATH):
    os.remove(RESULTS_PATH)
!{qa_cmd} 2>&1 | tee {LOG_DIR}/qa.log
assert os.path.isfile(RESULTS_PATH), "Soru-cevap betiği başarısız: logs/qa.log dosyasını kontrol edin"
qa_out = json.load(open(RESULTS_PATH, encoding="utf-8"))
assert len(qa_out["results"]) == len(QUESTIONS)
display(pd.Series(qa_out["summary"]).to_frame("değer"))

In [ ]:
pd.set_option("display.max_colwidth", 120)
qa_df = pd.DataFrame([{"tip": r.get("question_type"), "soru": r["question"], "altın cevap": r["answer"],
                       "MemGraphRAG cevabı": r["predicted_answer"]} for r in qa_out["results"]])
display(qa_df)
print("Örnek ham LLM yanıtı:\n", qa_out["results"][0]["raw_response"])

## 12. Değerlendirme: MemGraphRAG ve Klasik (Dense) RAG

Repodaki soru-cevap betiği cevapları **değerlendirmez**; yalnızca kaydeder. Bu yüzden repoda bulunan metrik sınıflarını (`src/evaluation/qa_eval.py`) kullanarak değerlendirme yapıyoruz:

- **EM (Exact Match)** ve **F1**: Normalize edilmiş cevap ile altın cevap arasındaki tam eşleşme ve token örtüşmesi (MRQA/HotpotQA resmi metrikleri).
- **Acc (içerme doğruluğu):** Normalize edilmiş altın cevap, model cevabında *tam kelime(ler) olarak* geçiyor mu? LLM cevapları genellikle cümle hâlinde olduğu için EM'den daha anlamlıdır.
- **Recall@k:** Sorunun her altın destekleyici pasajı için, pasaj gövdesinden en az bir 50 karakterlik parça erişilen ilk k chunk'ta geçiyor mu? Chunk'lar kayan 256 token'lık pencereler olduğu için pasajlarla birebir örtüşmez. Başlık eşleşmesi ise yanıltıcıdır, çünkü köprü sorularında ikinci pasajın başlığı çoğunlukla birinci pasajın içinde de geçer.

**Adil karşılaştırma:** Klasik RAG tabanı da *aynı* chunk'ları, *aynı* BGE gömmelerini, *aynı* LLM'i ve *aynı* QA istemini kullanır. Tek fark erişim yöntemidir: BGE ile en benzer `TOP_K` chunk (repodaki `dense_passage_retrieval`) ya da MemGraphRAG'ın olgu → graf → PPR erişimi. Karşılaştırılan dört yapılandırma:

| Yöntem | Olgu seçimi |
|---|---|
| Klasik RAG (dense) | Olgu yok; sorguya en benzer `TOP_K` chunk |
| MemGraphRAG (eşik 0.4) | README ayarı: ham benzerliği ≥ 0.4 olan **tüm** olgular (Bölüm 11'deki CLI çıktısı) |
| MemGraphRAG (eşik 0.6) | Repodaki `run_retrieval_test.sh` ayarı |
| MemGraphRAG (en iyi 5 olgu) | HippoRAG 2 tarzı: en benzer 5 olgu, eşik yalnızca bu 5 aday üzerinde (`use_raw_threshold_filter=False`, `linking_top_k=5`) |

Kod incelemesinde önemli bir ayrıntı bulundu: Repodaki BGE sarmalayıcısı, BGE'nin resmi yöntemi olan CLS havuzlaması yerine **ortalama havuzlama (mean pooling)** kullanıyor ve sorgu talimatlarını (*instruction*) yok sayıyor. Bu yüzden ham kosinüs benzerlikleri sıkışık ve yüksek çıkabiliyor; 0.4 gibi bir eşik neredeyse bütün olguları geçirebilir. Aşağıdaki ilk hücre bu dağılımı ölçer ve eşik seçimini yorumlamaya yardım eder.

In [ ]:
sys.path.insert(0, f"{REPO_DIR}/code")
import logging; logging.getLogger("httpx").setLevel(logging.WARNING)
from src.MemGraphRAG import MemGraphRAG
from src.utils.config_utils import BaseConfig
from src.utils.misc_utils import QuerySolution
from src.evaluation.qa_eval import QAExactMatch, QAF1Score
from src.utils.eval_utils import normalize_answer
import numpy as np

cfg = BaseConfig(save_dir=SAVE_DIR, llm_name=LLM_NAME, llm_base_url=LLM_BASE_URL, embedding_model_name=EMBEDDING_MODEL,
                 openie_mode="online", force_index_from_scratch=False, force_openie_from_scratch=False,
                 skip_fact_rerank=True, use_raw_threshold_filter=True, fact_similarity_threshold=FACT_THRESHOLD,
                 retrieval_top_k=TOP_K, qa_top_k=TOP_K, linking_top_k=50, passage_node_weight=0.05, damping=0.5,
                 max_new_tokens=1000, retrieval_max_workers=1, qa_max_workers=5)
rag = MemGraphRAG(global_config=cfg)
rag.prepare_retrieval_objects()
q_texts = [q["question"] for q in QUESTIONS]
golds = [[q["answer"]] for q in QUESTIONS]
gold_titles = [q["evidence"] for q in QUESTIONS]
rag.get_query_embeddings(q_texts)
print(f"Graf: {rag.graph.vcount()} düğüm, {rag.graph.ecount()} kenar | erişilebilir olgu: {len(rag.fact_node_keys)}")

# Soru–olgu ham benzerlik dağılımı: eşik değerinin kaç olguyu geçirdiğini gösterir
dist = []
for q in q_texts:
    s = rag.get_fact_scores(q, normalize=False)
    dist.append({"min": s.min(), "medyan": np.median(s), "maks": s.max(),
                 "≥0.4 oranı": (s >= 0.4).mean(), "≥0.6 oranı": (s >= 0.6).mean()})
print("\nSoru–olgu ham kosinüs benzerliği (sorular üzerinden ortalama):")
display(pd.DataFrame(dist).mean().round(3).to_frame("ortalama"))

In [ ]:
def chunk_has_passage(doc, title, w=50):
    # Altın pasajın GÖVDESİNDEN 50 karakterlik bir parça chunk'ta geçiyor mu? (başlık eşleşmesi yanıltıcıdır:
    # köprü sorularında ikinci pasajın başlığı genellikle birinci pasajın metninde de geçer)
    d = " ".join(doc.lower().split())
    body = " ".join(passages[title].lower().split())
    return any(body[i:i + w] in d for i in range(0, max(1, len(body) - w + 1), w))

def recall_at_k(docs_list, titles_list, k):
    scores = [sum(any(chunk_has_passage(d, t) for d in docs[:k]) for t in titles) / len(titles)
              for docs, titles in zip(docs_list, titles_list)]
    return sum(scores) / len(scores)

contains = lambda g, p: f" {normalize_answer(g)} " in f" {normalize_answer(p or '')} "   # kelime düzeyinde içerme

def evaluate(name, predictions, docs_list):
    predictions = [p or "" for p in predictions]
    em = QAExactMatch(cfg).calculate_metric_scores(golds, predictions)[0]["ExactMatch"]
    f1 = QAF1Score(cfg).calculate_metric_scores(golds, predictions)[0]["F1"]
    acc = sum(contains(g[0], p) for g, p in zip(golds, predictions)) / len(golds)
    return {"yöntem": name, "EM": em, "F1": f1, "Acc": acc,
            "Recall@2": recall_at_k(docs_list, gold_titles, 2), f"Recall@{TOP_K}": recall_at_k(docs_list, gold_titles, TOP_K)}

def run_dense_rag():
    sols = []
    for q in q_texts:
        ids, scores = rag.dense_passage_retrieval(q)
        docs = [rag.chunk_embedding_store.get_row(rag.passage_node_keys[i])["content"] for i in ids[:TOP_K]]
        sols.append(QuerySolution(question=q, docs=docs, doc_scores=scores[:TOP_K]))
    return rag.qa(sols)

def run_memgraphrag(threshold, raw_filter=True, linking_top_k=50):
    gc = rag.global_config
    old = (gc.fact_similarity_threshold, gc.use_raw_threshold_filter, gc.linking_top_k)
    gc.fact_similarity_threshold, gc.use_raw_threshold_filter, gc.linking_top_k = threshold, raw_filter, linking_top_k
    try:
        return rag.qa(rag.retrieve(q_texts))
    finally:
        gc.fact_similarity_threshold, gc.use_raw_threshold_filter, gc.linking_top_k = old

# 1) MemGraphRAG (eşik 0.4): repo betiğinin (Bölüm 11) çıktısı
mem_preds = [r["predicted_answer"] for r in qa_out["results"]]
mem_docs = [r["retrieved_docs"] for r in qa_out["results"]]
# 2) Klasik dense RAG
dense_sols, dense_raw, _ = run_dense_rag()
# 3) MemGraphRAG (eşik 0.6, repodaki run_retrieval_test.sh değeri)
mem06_sols, _, _ = run_memgraphrag(0.6)
# 4) MemGraphRAG, HippoRAG 2 tarzı: en benzer 5 olgu (eşik filtresi yalnızca bu 5 aday üzerinde)
memtop5_sols, _, _ = run_memgraphrag(FACT_THRESHOLD, raw_filter=False, linking_top_k=5)

eval_df = pd.DataFrame([
    evaluate("Klasik RAG (dense, BGE top-k)", [s.answer for s in dense_sols], [s.docs for s in dense_sols]),
    evaluate(f"MemGraphRAG (eşik {FACT_THRESHOLD})", mem_preds, mem_docs),
    evaluate("MemGraphRAG (eşik 0.6)", [s.answer for s in mem06_sols], [s.docs for s in mem06_sols]),
    evaluate("MemGraphRAG (en iyi 5 olgu)", [s.answer for s in memtop5_sols], [s.docs for s in memtop5_sols]),
]).set_index("yöntem").round(3)
display(eval_df)
eval_df.plot.bar(figsize=(10, 4), rot=0, title=f"HotpotQA alt kümesi ({len(QUESTIONS)} soru) – {LLM_NAME}")
plt.ylim(0, 1.05); plt.legend(loc="upper left", ncol=5, fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# Soru tipine göre (bridge = köprü/çok adımlı, comparison = karşılaştırma) içerme doğruluğu
types = [q["question_type"] for q in QUESTIONS]
acc_of = lambda preds: [contains(g[0], p) for g, p in zip(golds, preds)]
by_type = pd.DataFrame({"tip": types,
                        "Klasik RAG": acc_of([s.answer for s in dense_sols]),
                        "MemGraphRAG": acc_of(mem_preds)}).groupby("tip").mean().round(3)
by_type["soru sayısı"] = pd.Series(types).value_counts()
display(by_type)

compare = pd.DataFrame({"soru": q_texts, "altın": [g[0] for g in golds],
                        "Klasik RAG": [s.answer for s in dense_sols], "MemGraphRAG": mem_preds})
diff = compare[[a != b for a, b in zip(acc_of(compare["Klasik RAG"]), acc_of(compare["MemGraphRAG"]))]]
print(f"İki yöntemin doğruluk açısından ayrıştığı {len(diff)} soru:")
display(diff)

## 13. Tek Bir Soru Üzerinde Adım Adım MemGraphRAG Erişimi

Aşağıdaki hücre, `retrieve()` fonksiyonunun içindeki adımları tek bir soru için açar:

1. Olgu skorları ve eşik filtresi
2. Tohum varlıklar
3. PPR ile pasaj sıralaması (klasik dense sıralamayla karşılaştırmalı)
4. LLM cevabı

Soru olarak, iki yöntemin erişim başarısının farklı olduğu bir soru seçilir.

In [ ]:
import numpy as np

def pick_question():
    for i, q in enumerate(q_texts):
        if recall_at_k([mem_docs[i]], [gold_titles[i]], TOP_K) != recall_at_k([dense_sols[i].docs], [gold_titles[i]], TOP_K):
            return i
    return 0

qi = pick_question(); q = q_texts[qi]
print(f"Soru: {q}\nAltın cevap: {golds[qi][0]}\nAltın pasajlar: {gold_titles[qi]}\n")

raw = rag.get_fact_scores(q, normalize=False)
norm = rag.get_fact_scores(q, normalize=True)
idx, facts, log = rag.rerank_facts(q, norm)
fact_rows = rag.fact_embedding_store.get_rows(rag.fact_node_keys)
order = np.argsort(-raw)[:10]
display(pd.DataFrame([{"olgu": fact_rows[rag.fact_node_keys[j]]["content"], "ham benzerlik": round(float(raw[j]), 3),
                       "eşiği geçti mi": bool(raw[j] >= rag.global_config.fact_similarity_threshold)} for j in order]))
print(f"Eşiği geçen olgu sayısı: {len(facts)} / {len(raw)}")

phrase_scores = collections.defaultdict(list)
for rank, f in enumerate(facts):
    for p in (f[0], f[2]):
        phrase_scores[p.lower()].append(float(norm[idx[rank]]))
seeds = sorted(((np.mean(v), p) for p, v in phrase_scores.items()), reverse=True)[:10]
print("En güçlü tohum varlıklar:", [p for _, p in seeds])

if facts:
    ppr_ids, ppr_scores = rag.graph_search_with_fact_entities(q, link_top_k=50, query_fact_scores=norm, top_k_facts=facts,
                                                              top_k_fact_indices=idx, passage_node_weight=0.05)
else:
    ppr_ids, ppr_scores = rag.dense_passage_retrieval(q)
dense_ids, _ = rag.dense_passage_retrieval(q)
content = lambda i: rag.chunk_embedding_store.get_row(rag.passage_node_keys[i])["content"]
has_gold = lambda txt: any(chunk_has_passage(txt, t) for t in gold_titles[qi])
display(pd.DataFrame({"sıra": range(1, TOP_K + 1),
                      "MemGraphRAG (PPR)": [content(i)[:90] + ("  ✅" if has_gold(content(i)) else "") for i in ppr_ids[:TOP_K]],
                      "Klasik RAG (dense)": [content(i)[:90] + ("  ✅" if has_gold(content(i)) else "") for i in dense_ids[:TOP_K]]}))
print("✅ = altın pasajın metnini içeren chunk")
print("\nMemGraphRAG cevabı:", mem_preds[qi], "\nKlasik RAG cevabı :", dense_sols[qi].answer)

## 14. Karşılaşılan Hatalar ve Çözümleri

Bu notebook hazırlanırken repo, Colab'ın güncel ortamıyla aynı sürümlerde (Python 3.13.x, torch 2.11, transformers 5.x, numpy 2.x) çalıştırıldı. Karşılaşılan hatalar ve uygulanan çözümler:

| # | Aşama | Hata mesajı / belirti | Kök neden | Çözüm |
|---|---|---|---|---|
| 1 | Kurulum | `ERROR: No matching distribution found for torchvision==0.20.1` | `requirements.txt` → `vllm==0.6.6.post1` → Python 3.13 için wheel'i olmayan eski torch/torchvision | Sabitlenmiş gereksinimleri kurmadık. Kod incelenerek gerçekten kullanılan hafif paketler kuruldu (Bölüm 5) |
| 2 | İçe aktarma | `ModuleNotFoundError: No module named 'gritlm'` | GritLM koşulsuz içe aktarılıyor | Yama **P1**: opsiyonel içe aktarma |
| 3 | Veri | `FileNotFoundError: ... datasets/corpus/corpus.txt` | README'deki dosyalar repoda yok; repo `dataset/` klasörü kullanıyor | HotpotQA'dan README formatında `corpus.txt` ve `small-questions.json` üretildi (Bölüm 8) |
| 4 | Betikler | `code/run_index.sh: line 32: PYTHON: unbound variable` | `set -u` + tanımsız `$PYTHON`, sabit `/home/wcj/...` yolları, `--llm-base-url ""` | Betikler yerine `python code/index.py ...` doğrudan, parametreleriyle çağrıldı |
| 5 | İndeksleme | `RuntimeError: index_with_memory failed during schema extraction: Expecting value: line 1 column 1 (char 0)` | Bellek aşamalarında JSON olmayan tek bir LLM yanıtı tüm akışı durduruyor; yanıt önbellekte kaldığı için hata kalıcı | Yama **P2** (dayanıklı ayrıştırma) |
| 6 | Erişim | `AssertionError` (`get_top_k_weights`) | Tüm olgular eşiği geçtiğinde min-max normalizasyonu bir olguya 0 ağırlık veriyor; değiştirilmiş olgular normalize edilmediğinde düğüm bulunamıyor | Yamalar **P6** ve **P5** |
| 7 | LLM sunucusu | Uzun istemlerde bozuk JSON | Ollama'nın T4 için varsayılan bağlamı 4096 token; uzun istemlerin başı sessizce kesiliyor | `OLLAMA_CONTEXT_LENGTH=16384` ve `--conflict-max-related 10` |
| 8 | LLM sunucusu | Ollama kurulumunda `requires zstd for extraction` | Güncel Ollama sürümleri `.tar.zst` arşivi olarak dağıtılıyor | Kurulumdan önce `apt-get install zstd` |
| 9 | Kalite | LLM'e giden pasajlar tamamen küçük harf | BGE tokenizer'ı uncased; `index.py` chunk'ları bu tokenizer ile decode ediyor | `--tokenizer Qwen/Qwen2.5-7B-Instruct` (kayıpsız BPE) |
| 10 | Mantık | Çelişki güven eşiği atlanıyor; uydurulmuş olgu kimlikleri ilgisiz olguları silebiliyor | `conflicting_triple_ids` ve `triple_id` doğrulanmıyor | Yamalar **P3** ve **P4** |
| 11 | Kalite | Olgu eşiği (0.4) neredeyse tüm olguları geçiriyor | BGE sarmalayıcısı mean pooling kullanıyor ve sorgu talimatını yok sayıyor; benzerlikler yüksek ve sıkışık | Dağılım ölçüldü; eşik 0.6 ve "en iyi 5 olgu" varyantlarıyla karşılaştırıldı (Bölüm 12) |
| 12 | Yapılandırma | OpenAI ile `--llm-base-url ""` → tüm istekler başarısız; `127.0.0.1` adresinde `Missing credentials` | Boş string `None` sayılmıyor; sahte anahtar yalnızca URL'de `localhost` geçerse atanıyor | URL her zaman tam verildi (`https://api.openai.com/v1`, `http://localhost:11434/v1`) ve `OPENAI_API_KEY` açıkça ayarlandı |

Hatalardan bazıları aynı ortamda çalıştırılarak birebir alındı: 1, 2, 3, 4 ve 6 gerçek veriyle; 5 ise JSON olmayan bir yanıt simüle edilerek. 7, 8 ve 9 Ollama/BGE belgeleri ve kod incelemesiyle doğrulanıp önceden önlendi.

Aşağıdaki hücre **bu çalıştırmada** loglara yansıyan dayanıklılık olaylarını sayar. Örneğin P2 yaması sayesinde atlanan bozuk LLM yanıtları burada görünür.

In [ ]:
index_log = open(f"{LOG_DIR}/index.log", encoding="utf-8", errors="ignore").read()
qa_log = open(f"{LOG_DIR}/qa.log", encoding="utf-8", errors="ignore").read()
print("P2 ile atlanan bozuk bellek yanıtı sayısı :", index_log.count("Unparseable memory-stage LLM output"))
print("Varlık çıkarılamayan (NER hatalı) chunk sayısı:", sum(1 for d in docs if not d["extracted_entities"]))
print("Üçlü çıkarılamayan chunk sayısı          :", empty)
for line in qa_log.splitlines():
    if "Fact reranking stats" in line or "Fact filtering stats" in line:
        print(line.split("|")[-1].strip())
if LLM_BACKEND == "ollama":
    !grep -iE "error|truncat" {LOG_DIR}/ollama.log | tail -n 5 || true

## 15. Teknik Değerlendirme: MemGraphRAG, Klasik RAG ve GraphRAG

### 15.1 Bu çalıştırmanın maliyet özeti

MemGraphRAG'ın indeksleme maliyeti, klasik RAG'a göre en belirgin farkıdır. Aşağıdaki hücre LLM önbelleğinden (SQLite) gerçek çağrı ve token sayılarını hesaplar.

In [ ]:
import sqlite3
con = sqlite3.connect(f"{SAVE_DIR}/llm_cache/{LLM_LABEL}_cache.sqlite")
meta = [json.loads(m) for (m,) in con.execute("SELECT metadata FROM cache")]
con.close()
n_calls = len(meta)
p_tok = sum(m.get("prompt_tokens", 0) for m in meta)
c_tok = sum(m.get("completion_tokens", 0) for m in meta)
n_chunks, n_facts0 = len(docs), stage_df["olgu"].iloc[0]
# İndeksleme çağrıları: chunk başına NER + üçlü, ilk olgu başına şema, çelişki tespit görevleri, çözüm bileşenleri
idx_calls = 2 * n_chunks + int(n_facts0) + det["summary"]["num_detection_tasks"] + res["summary"]["num_components"]
summary = pd.Series(dtype=object, data={
    "chunk sayısı": n_chunks,
    "ilk olgu sayısı": int(n_facts0),
    "nihai olgu sayısı": int(stage_df["olgu"].iloc[-1]),
    "indeksleme LLM çağrısı": idx_calls,
    "≈ chunk başına indeksleme çağrısı": round(idx_calls / max(n_chunks, 1), 1),
    "önbellekteki toplam benzersiz LLM çağrısı (indeksleme + tüm QA koşuları)": n_calls,
    "girdi token": p_tok, "çıktı token": c_tok,
    "indeksleme süresi (dk, bu çalıştırma; önbellek isabetleri dahil)": round(INDEX_SECONDS / 60, 1),
    "aynı iş yükünün GPT-4o-mini maliyeti (USD, $0.15/$0.60 per 1M)": round(p_tok * 0.15e-6 + c_tok * 0.60e-6, 3),
})
display(summary.to_frame("değer"))

### 15.2 Karşılaştırma tablosu

| Boyut | Klasik (dense) RAG | Microsoft GraphRAG | HippoRAG 2 | **MemGraphRAG** |
|---|---|---|---|---|
| İndeks birimi | Metin parçaları + vektörler | Varlık/ilişki grafı + Leiden toplulukları + LLM topluluk özetleri | OpenIE üçlülerinden varlık–pasaj grafı | **Üç katmanlı bellek** (şema / olgu / pasaj) → tip–varlık–pasaj grafı |
| Çıkarım bakışı | — | Her chunk bağımsız | Her chunk bağımsız | Chunk çıkarımı + **global bellek** üzerinde ortak küratörlük |
| Gürültü/çelişki kontrolü | Yok | Yok (özetlerde birleştirme) | Yok (tüm üçlüler grafa girer) | **Ontoloji filtreleme + LLM ile çelişki tespiti ve kanıta dayalı çözüm** |
| Soyutlama | Yok | Topluluk özetleri (metin) | Yok | **Şema/tip katmanı** (ontoloji çıkarımı) |
| Erişim | Vektör benzerliği (top-k) | Yerel: varlık komşuluğu; Global: özetler üzerinde map-reduce | Olgu eşleme + tanıma filtresi + PPR | HippoRAG 2 ile aynı: olgu eşleme + eşik + **PPR** (tip düğümleri üzerinden de yayılım) |
| Güçlü olduğu sorular | Tek adımlı, yerel olgu soruları | Korpus geneli "özetle/temalar neler" soruları | Çok adımlı (multi-hop) sorular | Çok adımlı sorular + tutarlılık gerektiren olgusal sorular |
| İndeksleme maliyeti | Çok düşük (yalnızca gömme) | Çok yüksek (çıkarım + topluluk özetleri) | Orta (chunk başına 2 LLM çağrısı) | **Yüksek**: chunk başına 2 çağrı + **olgu başına 1 şema çağrısı** + çelişki çağrıları |
| Sorgu maliyeti | 1 LLM çağrısı | Global modda çok sayıda çağrı | 1–2 LLM çağrısı | 1 LLM çağrısı (yeniden sıralama kapalıyken) |

### 15.3 Değerlendirme

**Klasik RAG'dan farkı.** Klasik RAG, soruya *yüzeysel olarak benzeyen* parçaları getirir. Çok adımlı bir soruda (ör. "The Newcomers filminin yıldızlarından biri ne ile tanınır?") ikinci adımın pasajı (Chris Evans) soruda geçmez ve bu yüzden düşük skor alabilir. MemGraphRAG önce soruyla ilgili **olguları** bulur, örneğin `(the newcomers, starring, chris evans)`. Bu olgudaki varlıklardan başlayan PPR yürüyüşü, soruda hiç geçmeyen "Chris Evans" pasajına graf üzerinden ulaşır. Bölüm 12'deki **Recall@k** ve **bridge** tipi soru sonuçları bu etkinin bu küçük veri kümesinde ne kadar ortaya çıktığını gösterir. Bölüm 13'te tek bir soru üzerinden de görülebilir. Bunun bedeli indeksleme maliyetidir. Klasik RAG yalnızca gömme hesaplar; MemGraphRAG ise chunk başına onlarca LLM çağrısı yapar (Bölüm 15.1). Erişim kalitesi büyük ölçüde OpenIE ve ontoloji çıktısının kalitesine, dolayısıyla seçilen LLM'e bağlıdır.

**GraphRAG'dan (Microsoft) farkı.** GraphRAG grafı topluluklara ayırır ve her topluluk için LLM ile özet yazar. Özetler korpus geneli "anlamlandırma" sorularında güçlüdür, ancak tek tek olguların doğruluğunu ya da tutarlılığını denetlemez. Grafa giren çelişkili veya gürültülü ilişkiler özetlere de taşınır. MemGraphRAG'da topluluk tespiti ve özet yoktur. Soyutlamayı **şema (tip) katmanıyla** yapar ve grafı kurmadan önce **olgu düzeyinde küratörlük** uygular: seyrek şemaları eler, sert çelişkileri pasaj kanıtıyla çözer. Bu, makalenin "chunk düzeyinde izole çıkarım → gürültülü/tutarsız graf" eleştirisinin doğrudan cevabıdır.

**HippoRAG 2'den farkı.** Kod incelemesi, erişim tarafının (olgu eşleme, tanıma belleği filtresi, PPR, pasaj ağırlığı 0.05, sönüm 0.5) HippoRAG 2'den neredeyse aynen alındığını gösteriyor. Fark tamamen indeksleme tarafında: ontoloji çıkarımı ve filtreleme, çelişki tespiti ve çözümü, ek **tip düğümleri** ve yalnızca küratörlükten geçen olgulardan kurulan graf. Dolayısıyla MemGraphRAG'ın kazancı "daha temiz bir graf"tan gelir.

**Gözlenen zayıf yönler ve sınırlamalar**

- **Maliyet ölçeklenmesi:** Şema çıkarımı *olgu başına* bir LLM çağrısı yapar. Olgu sayısı chunk sayısının ~10–15 katı olduğundan, büyük korpuslarda baskın maliyet budur. Repo bu çağrıları toplu (batch) göndermez.
- **Sağlamlık:** Orijinal kod tek bir bozuk JSON yanıtında tüm indekslemeyi durduruyordu. Küçük açık kaynak modellerle bu risk gerçektir (P2).
- **Hiperparametre hassasiyeti:** Olgu eşiği (0.4 / 0.6) erişimi doğrudan değiştirir. Eşik çok yüksekse hiçbir olgu geçmez ve sistem sessizce klasik dense erişime düşer. Çok düşükse tohum kümesi gürültülü olur. Repodaki BGE sarmalayıcısının mean pooling kullanması ve sorgu talimatını yok sayması benzerlikleri yükselttiği için eşiğin veri ve modele göre ayarlanması gerekir. Bölüm 12'deki benzerlik dağılımını, dört varyantın sonuçlarını ve Bölüm 14'teki "Fact reranking stats" satırını bu açıdan yorumlayın.
- **Dil:** `text_processing` İngilizce dışındaki karakterleri sildiği için (`İstanbul` → `i stanbul`) Türkçe gibi diller için normalizasyonun değiştirilmesi gerekir.
- **Küçük ölçekli deney:** 20 soruluk alt kümede yöntemler arasındaki farklar istatistiksel olarak kesin değildir. Makaledeki sonuçları doğrulamak için tam veri kümesi ve daha güçlü bir LLM (ör. GPT-4o-mini) gerekir.

> **Not (öğrenci için):** Bölüm 12–15'teki tablolar çalıştırmaya göre değişir. Teslimden önce bu bölümdeki yorumları kendi sayılarınıza göre bir-iki cümleyle güncellemeniz önerilir. Örneğin hangi yöntem daha yüksek Recall/Acc aldı, kaç olgu elendi, kaç çelişki çözüldü?

## 16. Sonuç

- MemGraphRAG, Colab'ın güncel ortamında (Python 3.13, T4 GPU) **açık kaynak bir LLM (Qwen2.5-7B-Instruct, Ollama)** ve **BGE-large** gömme modeliyle uçtan uca çalıştırıldı. Repodaki iki adımlı örnek (indeksleme → erişim/QA) HotpotQA alt kümesinde tamamlandı.
- Reponun Colab'da çalışması için 4 ortam/yapılandırma sorunu (gereksinimler, eksik veri dosyaları, bozuk betikler, Ollama bağlamı) aşıldı ve kaynak koda 8 küçük yama uygulandı. Yamalar 4 çökme hatasını, 3 mantık hatasını ve 1 log sorununu giderir.
- Sonuçlar klasik dense RAG ile aynı koşullarda karşılaştırıldı. MemGraphRAG'ın üç katmanlı belleği, ontoloji filtrelemesi ve çelişki çözümü ara çıktılar üzerinden somut olarak incelendi.

## Kaynaklar

1. Wu, C., Xiang, Z., Tang, Y., Chen, Z., Zhang, Q., Su, J. *MemGraphRAG: Memory-based Multi-Agent System for Graph Retrieval-Augmented Generation.* KDD 2026. [arXiv:2606.00610](https://arxiv.org/abs/2606.00610) · [GitHub](https://github.com/XMUDeepLIT/MemGraphRAG)
2. Gutiérrez, B. J. ve ark. *From RAG to Memory: Non-Parametric Continual Learning for Large Language Models (HippoRAG 2).* 2025.
3. Edge, D. ve ark. *From Local to Global: A Graph RAG Approach to Query-Focused Summarization (Microsoft GraphRAG).* 2024.
4. Yang, Z. ve ark. *HotpotQA: A Dataset for Diverse, Explainable Multi-hop Question Answering.* EMNLP 2018.
5. Qwen Team. *Qwen2.5 Technical Report.* 2024 · [Ollama](https://ollama.com) · [BAAI/bge-large-en-v1.5](https://huggingface.co/BAAI/bge-large-en-v1.5)